In [6]:
from ..structure_output import *

load_dotenv(override=True)

DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')
model=init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek-v4-flash',
    model_provider='deepseek',
    #flash思考模式不支持结构化输出
    extra_body={"thinking":{"type":"disabled"}}
)

In [5]:

#列表提取+类型嵌套(建议嵌套结构<3层)+#pydantic模型限制条件
class student(BaseModel):
    name: str=Field(
        description='学生姓名'
    )
    age: int=Field(
        description='年龄',
        ge=0,#年龄范围大于0
        le=100#年龄范围小于100
    )

class school(BaseModel):
    school_name: str=Field(
        description='学校名',
        min_length=3,#限制名字长度
        max_length=5
    )
    teacher:str=Field(
        description='学校老师'
    )
    students:List[student]=Field(
        description='学生列表'
    )
school_model=model.with_structured_output(school)
school=school_model.invoke('中心小学有一名王老师，带了张三，李四两个学生,分别13和14岁')
print(school)

school_name='中心小学' teacher='王老师' students=[student(name='张三', age=13), student(name='李四', age=14)]


In [19]:
print('验证输出模型有效性')
try:
    school=school_model.invoke('1122233小学有一名李老师，带了张三，李四两个学生，分别130,140岁')
    print(f'{school}\n检验没有问题')
except ValidationError as e:
    for error in e.errors():
        print(f"校验失败\n{error['loc']}{error['msg']},but your input is {error['input']}")

验证输出模型有效性
校验失败
('school_name',)String should have at most 5 characters,but your input is 1122233小学
校验失败
('students', 0, 'age')Input should be less than or equal to 100,but your input is 130
校验失败
('students', 1, 'age')Input should be less than or equal to 100,but your input is 140
